# 🏎️ PyRacer on Kaggle — Teach a Car to Drive with Deep Q-Learning

This notebook trains the **PyRacer** reinforcement-learning agent end-to-end on
Kaggle. A car learns to drive procedurally generated tracks from raycast sensors
using a **Double DQN** (the default; a self-supervised *JEPA* world-model variant
also lives in the repo).

**What you'll do here**
1. Bootstrap the repo + dependencies (works headless — no display needed).
2. Inspect the **state** (11 numbers) and **action** space (5 moves).
3. Train a DQN agent and watch the **learning curve** climb.
4. Visualise a **greedy rollout** of the trained car on the track.

> **Before you run:** open the notebook settings (right sidebar) and switch
> **Internet = On** (needed to clone the repo + `pip install pygame`).
> A GPU is *optional* — the network is tiny, so CPU is plenty fast.

> Everything runs **headless**: we set the SDL driver to `dummy`, so PyGame never
> tries to open a window.

## 1. Setup — bootstrap the repo (headless)

We force PyGame into a windowless "dummy" mode *before* importing it, install the
one missing dependency (`pygame`), then clone the PyRacer source so we can import
its `game/` and `rl/` modules.

In [ ]:
import os, sys, subprocess

# Headless PyGame: no display/audio device on Kaggle. Must be set BEFORE pygame import.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

# torch / numpy / matplotlib already ship on Kaggle; pygame does not.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygame"], check=True)

# Clone the repo once, then make it importable.
REPO_URL = "https://github.com/StuehlerM/PyRacer.git"
REPO_DIR = "/kaggle/working/PyRacer"
if not os.path.isdir(os.path.join(REPO_DIR, "game")):
    # Fall back to the current dir when running outside Kaggle.
    target = REPO_DIR if os.path.isdir("/kaggle/working") else "PyRacer"
    if not os.path.isdir(os.path.join(target, "game")):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, target], check=True)
    REPO_DIR = target

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working dir:", os.getcwd())
print("Repo contents:", sorted(os.listdir(REPO_DIR))[:12])

## 2. Imports & reproducibility

Import the project's environment + agent, then seed every RNG (Python, NumPy,
PyTorch) so a re-run behaves the same.

In [ ]:
import random
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

from utils.config import config
from game.track import Track
from game.environment import RacingEnv
from rl.agent import DQNAgent, RandomAgent

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device = {device}")
print(f"STATE_DIM = {config.STATE_DIM}  (7 sensors + speed + sin + cos + progress)")
print(f"ACTION_DIM = {config.ACTION_DIM}")

## 3. Hyperparameters

Everything you'd want to tweak in one place. Defaults are sized to finish
comfortably inside a Kaggle session while still learning a visibly better policy.
Raise `EPISODES` for a stronger driver.

In [ ]:
SEED        = 42       # reproducible track + training
APPROACH    = "dqn"    # default learning approach (the repo also has "jepa")
EPISODES    = 400      # bump to 1000+ for a stronger policy (takes longer)
DUELING     = False    # set True for the Dueling DQN architecture
EVAL_WINDOW = 50       # moving-average window for "best model" selection
SAVE_PATH   = "/kaggle/working/best_model.pth" if os.path.isdir("/kaggle/working") else "best_model.pth"

set_seed(SEED)
print(f"approach={APPROACH}  episodes={EPISODES}  dueling={DUELING}")

## 4. The environment: state & actions

`RacingEnv` is a Gym-style wrapper (`reset()` / `step(action)` →
`(next_state, reward, done, info)`).

**State (11 floats, all normalised 0–1):** 7 raycast distances at
`[-90°,-60°,-30°,0°,30°,60°,90°]`, plus normalised speed, `sin(heading)`,
`cos(heading)`, and track progress.

**Actions (5 discrete):**

| # | Throttle | Steering | Meaning |
|---|----------|----------|---------|
| 0 | 1.0  | 0.0  | Full throttle straight |
| 1 | 0.8  | -0.8 | Accelerate + turn left |
| 2 | 0.8  | 0.8  | Accelerate + turn right |
| 3 | 0.3  | 0.0  | Coast straight |
| 4 | -1.0 | 0.0  | Brake hard |

Let's sanity-check with a few random steps.

In [ ]:
demo_env = RacingEnv(track=Track(seed=SEED), render=False)
s = demo_env.reset()
print("state shape:", s.shape)
print("initial state:", np.round(s, 3))

total = 0.0
for _ in range(50):
    s, r, done, info = demo_env.step(np.random.randint(config.ACTION_DIM))
    total += r
    if done:
        break
print(f"random 50-step reward: {total:.2f} | progress: {info['progress']*100:.1f}%")
demo_env.close()

## 5. Train the DQN agent

The classic DQN loop: **act → store transition → sample a replay batch → update
the Q-network**. Exploration (`epsilon`) decays as the agent gathers experience;
a target network is soft-updated for stability. We keep the best model by moving
-average reward.

In [ ]:
set_seed(SEED)
env = RacingEnv(track=Track(seed=SEED), render=False)
agent = DQNAgent(
    state_dim=config.STATE_DIM,
    action_dim=config.ACTION_DIM,
    use_dueling=DUELING,
    use_double_dqn=True,
)

reward_hist, avg_hist = [], []
best_avg = -float("inf")
window = []
start = time.time()

for ep in range(1, EPISODES + 1):
    state = env.reset()
    done, ep_reward, ep_loss, n_loss = False, 0.0, 0.0, 0
    while not done:
        action = agent.select_action(state, explore=True)
        next_state, reward, done, info = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        loss = agent.update()
        if loss is not None:
            ep_loss += loss; n_loss += 1
        agent.on_env_step()          # decays epsilon per env step
        state = next_state
        ep_reward += reward
    agent.increment_episode()

    reward_hist.append(ep_reward)
    window.append(ep_reward)
    if len(window) > EVAL_WINDOW:
        window.pop(0)
    avg = float(np.mean(window))
    avg_hist.append(avg)
    if avg > best_avg:
        best_avg = avg
        agent.save(SAVE_PATH)        # checkpoint the best policy so far

    if ep % 20 == 0 or ep == 1:
        mean_loss = ep_loss / max(1, n_loss)
        print(f"ep {ep:4d}/{EPISODES} | reward {ep_reward:8.2f} | "
              f"avg{EVAL_WINDOW} {avg:8.2f} | eps {agent.epsilon:.3f} | "
              f"loss {mean_loss:.4f} | best {best_avg:.2f}")

env.close()
print(f"\nDone in {time.time()-start:.1f}s. Best avg reward: {best_avg:.2f}")
print(f"Best model saved to: {SAVE_PATH}")

## 6. Learning curve

Per-episode reward (faint) with the moving average (bold). A rising trend means
the agent is making real progress around the track instead of crashing early.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(reward_hist, alpha=0.3, label="episode reward")
plt.plot(avg_hist, linewidth=2, label=f"moving avg ({EVAL_WINDOW})")
plt.xlabel("episode"); plt.ylabel("reward")
plt.title("PyRacer DQN — learning curve")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 7. Watch the trained agent

No display on Kaggle — so we run a **greedy** rollout (no exploration), record the
car's path, and draw it over the track with matplotlib. We load the *best* saved
checkpoint first, then compare an untrained-style random path against the learned
one.

In [ ]:
def rollout(agent, track, explore=False, max_steps=2000):
    # Run one greedy/random episode; return the car's (x, y) path and final info.
    env = RacingEnv(track=track, render=False, max_steps=max_steps)
    state = env.reset()
    xs = [float(env.game.car.position[0])]
    ys = [float(env.game.car.position[1])]
    done, info = False, {}
    while not done:
        action = agent.select_action(state, explore=explore)
        state, _, done, info = env.step(action)
        xs.append(float(env.game.car.position[0]))
        ys.append(float(env.game.car.position[1]))
    env.close()
    return xs, ys, info

def draw_track(ax, track):
    outer = np.array(track.outer_boundary)
    inner = np.array(track.inner_boundary)
    ax.plot(outer[:, 0], outer[:, 1], color="#444", lw=1)
    ax.plot(inner[:, 0], inner[:, 1], color="#444", lw=1)
    sx, sy = track.start_position
    ax.scatter([sx], [sy], c="lime", s=60, zorder=5, label="start")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")

# Load the best checkpoint into a fresh agent.
best_agent = DQNAgent(state_dim=config.STATE_DIM, action_dim=config.ACTION_DIM,
                      use_dueling=DUELING)
best_agent.load(SAVE_PATH)

viz_track = Track(seed=SEED)
rx, ry, rinfo = rollout(RandomAgent(action_dim=config.ACTION_DIM), viz_track, explore=True)
tx, ty, tinfo = rollout(best_agent, viz_track, explore=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (xs, ys, info, name) in zip(
        axes,
        [(rx, ry, rinfo, "Random agent"), (tx, ty, tinfo, "Trained DQN")]):
    draw_track(ax, viz_track)
    ax.plot(xs, ys, color="#e8590c", lw=2, label="path")
    ax.set_title(f"{name} — progress {info.get('progress', 0)*100:.1f}%")
    ax.legend(loc="upper right")
plt.suptitle("Random vs. trained — same track")
plt.show()

### Optional: animate the trained rollout

Renders the trained car's path as an inline HTML5 animation. Skip if you just
want the static plots above.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(7, 6))
draw_track(ax, viz_track)
(line,) = ax.plot([], [], color="#e8590c", lw=2)
(dot,) = ax.plot([], [], "o", color="#1864ab", ms=8)
ax.set_title("Trained DQN — greedy rollout")

step = max(1, len(tx) // 200)   # cap to ~200 frames
fx, fy = tx[::step], ty[::step]

def animate(i):
    line.set_data(fx[:i + 1], fy[:i + 1])
    dot.set_data([fx[i]], [fy[i]])
    return line, dot

anim = animation.FuncAnimation(fig, animate, frames=len(fx), interval=50, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 8. Where to go next

- **Train longer:** raise `EPISODES` (1000–5000) for a smoother driver.
- **Architecture:** set `DUELING = True` for a Dueling DQN.
- **Reward / physics / sensors:** every knob lives in `utils/config.py`.
- **Self-supervised variant:** the repo also ships a **JEPA** world-model agent —
  on the command line it's `python train.py --approach jepa`. DQN is the default
  here because it learns from an explicit reward signal, which is easier to follow.

The best checkpoint is saved at `SAVE_PATH` — download it from the Kaggle
**Output** tab to keep your trained model.